<div style="font-size: 24px; line-height: 1.6;">

# Survivorship Bias: Missing Failures Are Still Evidence

## Missing rows and missing values are clues, not noise

![Survivorship Bias: Missing Failures Are Still Evidence](../images/survivorship_bias.png)

</div>

<div style="font-size: 24px; line-height: 1.6;">

## Takeaway

Functions introduced: `pd.read_json`, `isna`, `value_counts`, `pd.crosstab`, `unstack`, `fillna`, `dropna`, `pd.merge`, `duplicated`, `drop_duplicates`.

**Concept learned: missing rows and missing values are evidence. A model trained only on survivors learns the wrong population — model what is observed for everyone (survival) first, then model the rest conditionally.**

</div>

<div style="font-size: 24px; line-height: 1.6;">

### Imports

</div>

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['font.size'] = 16
plt.rcParams['figure.figsize'] = (8, 5)

<div style="font-size: 24px; line-height: 1.6;">

## The story

Revenue exists for surviving startups. Failed startups often have missing revenue, or vanish from the dataset entirely. If you only analyze survivors, you are studying winners and calling it the population.

</div>

<div style="font-size: 24px; line-height: 1.6; margin-top: 224px;">

## 1. Load the startup dataset

This file is JSON — common when data comes from a web API. We wrote it in *records* orientation (a list of row-dicts), so we pass `orient='records'` to `pd.read_json()` to read it back.

</div>

In [2]:
startups = pd.read_json("../data/startup_survivorship.json", orient="records")
startups.head()

,startup_id,sector,seed_funding_millions,market_score,survived_3yr,year3_revenue_millions
0,1,Health,0.36,-0.47,True,36.36
1,2,AI,0.70,0.12,True,28.05
2,3,Retail,0.13,0.40,False,NaN
3,4,Games,1.60,1.20,False,NaN
4,5,Health,0.62,0.24,True,97.69


In [3]:
startups.info()

<class 'pandas.DataFrame'>
RangeIndex: 2403 entries, 0 to 2402
Data columns (total 6 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   startup_id              2403 non-null   int64  
 1   sector                  2403 non-null   str    
 2   seed_funding_millions   2403 non-null   float64
 3   market_score            2403 non-null   float64
 4   survived_3yr            2403 non-null   bool   
 5   year3_revenue_millions  634 non-null    float64
dtypes: bool(1), float64(3), int64(1), str(1)
memory usage: 108.6 KB


<div style="font-size: 24px; line-height: 1.6; margin-top: 224px;">

## 2. Missingness map with `df.isna()`

`isna()` returns a True/False grid marking where values are absent, so you can see *which* rows lose revenue — the first step in asking whether it goes missing for a reason (the startup failed) rather than at random.

</div>

In [4]:
startups.isna().head()

,startup_id,sector,seed_funding_millions,market_score,survived_3yr,year3_revenue_millions
0,False,False,False,False,False,False
1,False,False,False,False,False,False
2,False,False,False,False,False,True
3,False,False,False,False,False,True
4,False,False,False,False,False,False


<div style="font-size: 24px; line-height: 1.6; margin-top: 224px;">

## 3. Missing counts with `df.isna().sum()`

Summing that True/False grid down each column turns the map into a tally, so you can see revenue is missing far more often than anything else — the clue that the missingness is tied to the outcome, not scattered evenly.

</div>

In [5]:
startups.isna().sum().sort_values(ascending=False)

year3_revenue_millions    1769
startup_id                   0
sector                       0
seed_funding_millions        0
market_score                 0
survived_3yr                 0
dtype: int64

<div style="font-size: 24px; line-height: 1.6; margin-top: 224px;">

## 4. Outcome counts

Survivorship analysis starts with the outcome distribution.

</div>

In [6]:
startups["survived_3yr"].value_counts()

survived_3yr
False    1769
True      634
Name: count, dtype: int64

<div style="font-size: 24px; line-height: 1.6; margin-top: 224px;">

## 5. Cross missingness with the outcome

Is revenue missing because the startup failed?

</div>

In [7]:
pd.crosstab(startups["survived_3yr"], startups["year3_revenue_millions"].isna(),
            rownames=["survived_3yr"], colnames=["revenue_missing"])

revenue_missing,False,True
survived_3yr,,
False,0,1769
True,634,0


<div style="font-size: 24px; line-height: 1.6; margin-top: 224px;">

## 6. Fill carefully with `df.fillna()`

Filling can be right or wrong depending on meaning. Filling missing revenue with 0 implies the startup earned nothing — but maybe revenue was simply unrecorded.

</div>

In [8]:
startups["year3_revenue_millions"].fillna(0).describe()

count    2403.000000
mean        5.359680
std        12.564282
min         0.000000
25%         0.000000
50%         0.000000
75%         4.560000
max       147.980000
Name: year3_revenue_millions, dtype: float64

<div style="font-size: 24px; line-height: 1.6; margin-top: 224px;">

## 7. Drop carefully with `df.dropna()`

Dropping missing revenue creates a **survivor-only** table.

</div>

In [9]:
survivors_only = startups.dropna(subset=["year3_revenue_millions"])

<div style="font-size: 24px; line-height: 1.6; margin-top: 224px;">

## 8. Compare before and after

Always compare shapes and group counts after row removal.

</div>

In [10]:
print("Full dataset shape:   ", startups.shape)
print("Survivors-only shape: ", survivors_only.shape)

print("\nSector counts (full):")
print(startups["sector"].value_counts())

print("\nSector counts (survivors only):")
print(survivors_only["sector"].value_counts())

Full dataset shape:    (2403, 6)
Survivors-only shape:  (634, 6)

Sector counts (full):
sector
Retail     614
AI         474
Fintech    471
Games      443
Health     401
Name: count, dtype: int64

Sector counts (survivors only):
sector
Retail     161
AI         142
Games      121
Health     108
Fintech    102
Name: count, dtype: int64


<div style="font-size: 24px; line-height: 1.6; margin-top: 224px;">

## 9. Survived vs. failed per sector with `unstack()`

`groupby(["sector", "survived_3yr"]).size()` counts rows for every (sector, outcome) pair, but returns a hard-to-read stacked Series; `unstack()` pivots the inner level into columns so you get a readable sector × outcome table. The `False` column is exactly the failures that `dropna` threw away.

</div>

In [11]:
startups.groupby(["sector", "survived_3yr"]).size().unstack()

survived_3yr,False,True
sector,,
AI,332,142
Fintech,369,102
Games,322,121
Health,293,108
Retail,453,161


<div style="font-size: 24px; line-height: 1.6; margin-top: 224px;">

## 10. The biased headline vs. the honest one

What does the headline 'average startup revenue' look like from each table?

</div>

In [12]:
biased = survivors_only["year3_revenue_millions"].mean()
honest_zero_fill = startups["year3_revenue_millions"].fillna(0).mean()
print(f"Survivor-only average revenue: {biased:.2f}M")
print(f"All startups (failed = 0):     {honest_zero_fill:.2f}M")

Survivor-only average revenue: 20.31M
All startups (failed = 0):     5.36M


<div style="font-size: 24px; line-height: 1.6; margin-top: 224px;">

## 11. Joining tables with `pd.merge()`

Joins can *create* survivorship bias. A press directory only writes about companies that are still alive — merge against it with the default `how='inner'` and every failed startup silently disappears. `how='left'` keeps the full population.

![Inner vs. left merge](../images/merge_inner_left.png)

</div>

In [13]:
press = survivors_only[["startup_id"]].copy()
press["press_article"] = "feature story"

inner = startups.merge(press, on="startup_id")
left = startups.merge(press, on="startup_id", how="left")
print("all startups:      ", startups.shape)
print("inner merge result:", inner.shape, " <- failures silently gone")
print("left merge result: ", left.shape)

all startups:       (2403, 6)
inner merge result: (634, 7)  <- failures silently gone
left merge result:  (2403, 7)


<div style="font-size: 24px; line-height: 1.6; margin-top: 224px;">

## 12. Find duplicates with `df.duplicated()`

`duplicated()` flags every row that is an exact copy of an earlier one — duplicates silently double-count whatever you sum or average.

</div>

In [14]:
startups.duplicated().sum()

np.int64(3)

<div style="font-size: 24px; line-height: 1.6; margin-top: 224px;">

## 13. Remove duplicates with `df.drop_duplicates()`

Remove duplicates only after checking what they represent.

</div>

In [15]:
clean = startups.drop_duplicates()
print(startups.shape, clean.shape)

(2403, 6) (2400, 6)


<div style="font-size: 24px; line-height: 1.6;">

## The danger of filling: imputation changes the statistics

Back to section 6 for a harder look. Filling is never free: every strategy *fabricates* 1,769 revenue values — far more fabricated data than the 634 real values — and each strategy distorts the statistics in a different way.

</div>

In [16]:
real_rev = survivors_only["year3_revenue_millions"]
zero_fill = startups["year3_revenue_millions"].fillna(0)
mean_fill = startups["year3_revenue_millions"].fillna(
    startups["year3_revenue_millions"].mean()
)

print("corr(seed_funding, revenue)")
print("  real (survivors only):", round(survivors_only["seed_funding_millions"].corr(real_rev), 3))
print("  after mean-fill:      ", round(startups["seed_funding_millions"].corr(mean_fill), 3))

print("\ncorr(market_score, revenue)")
print("  real (survivors only):", round(survivors_only["market_score"].corr(real_rev), 3))
print("  after zero-fill:      ", round(startups["market_score"].corr(zero_fill), 3))

print("\nspread of revenue (std)")
print("  real (survivors only):", round(real_rev.std(), 2))
print("  after mean-fill:      ", round(mean_fill.std(), 2))

corr(seed_funding, revenue)
  real (survivors only): 0.138
  after mean-fill:       0.076

corr(market_score, revenue)
  real (survivors only): -0.017
  after zero-fill:       0.129

spread of revenue (std)
  real (survivors only): 17.17
  after mean-fill:       8.81


<div style="font-size: 24px; line-height: 1.6;">

Each fill strategy commits a different crime:

- **Mean-fill destroys real relationships.** The seed-funding correlation is roughly cut in half, and the spread of revenue drops from about 17 to about 9 — injecting 1,769 identical values flattens every pattern in the column.
- **Zero-fill invents relationships that do not exist.** Among survivors, `market_score` has *no* relationship with revenue. But market score does predict *survival* — so writing 0 into every failed startup copies the survival pattern into the revenue column, and a correlation appears from nowhere. A model would happily learn it.

Every statistic you compute after filling is partly a statistic of your fill strategy. Imputation is a **modeling decision**, not cleanup — document it, justify it, and check what it does to the numbers you care about.

</div>

<div style="font-size: 24px; line-height: 1.6;">

## Don't change data silently

Prefer creating a new object over overwriting the original during EDA — your future self will thank you. That is why we wrote `survivors_only = startups.dropna(...)` above instead of overwriting `startups`.

</div>

<div style="font-size: 24px; line-height: 1.6;">

## Why this matters to a data scientist

Everything downstream inherits this bias. Train a revenue model on `survivors_only` and it does not learn *what makes startups succeed* — it learns *what successful startups look like*, which is a different question. Deployed on next year's cohort (which contains its future failures), its predictions will be systematically too optimistic, because the model has literally never seen a failure.

The deeper problem: the missingness here is **informative**. Revenue is missing *because* the startup failed — statisticians call this *missing not at random* (MNAR): the mechanism that hides a value is tied to the value itself. No fill strategy can recover information that was never recorded.

</div>

<div style="font-size: 24px; line-height: 1.6;">

## The solution: model what you actually observe

We cannot conjure the missing revenue, but we are not stuck — restructure the problem around what *is* observed for everyone:

1. **Model survival first.** `survived_3yr` is present for every row, so surviving-vs-failing is an honest question for the full population.
2. **Model revenue *given* survival** on the survivors — and label that estimate as conditional.
3. **Multiply the two** for an honest expected value (a *two-stage* or *hurdle* model).

And the strongest fix is upstream: **design the missingness out** — track cohorts from founding day so failures stay in the dataset, instead of scraping success stories after the fact.

</div>

In [17]:
p_survive = startups["survived_3yr"].mean()
revenue_if_survive = survivors_only["year3_revenue_millions"].mean()

print(f"Stage 1 - P(survive 3 years):    {p_survive:.1%}  (uses all {len(startups)} rows)")
print(f"Stage 2 - E[revenue | survived]: {revenue_if_survive:.2f}M  (uses {len(survivors_only)} survivors)")
print(f"\nHonest expected revenue:  {p_survive * revenue_if_survive:.2f}M per startup founded")
print(f"Survivor-only headline:   {revenue_if_survive:.2f}M  <- almost 4x too optimistic")

Stage 1 - P(survive 3 years):    26.4%  (uses all 2403 rows)
Stage 2 - E[revenue | survived]: 20.31M  (uses 634 survivors)

Honest expected revenue:  5.36M per startup founded
Survivor-only headline:   20.31M  <- almost 4x too optimistic


<div style="font-size: 24px; line-height: 1.6;">

This also explains why the zero-fill average in section 10 was defensible *here*: in this dataset, missing revenue really does mean "failed, earned nothing", so the zero-fill mean and the two-stage estimate agree. When missing instead means "unrecorded", zero-fill is simply wrong — but the two-stage decomposition still works, because it never pretends to know values it does not have.

</div>